In [ ]:
import pandas as pd
import os

try:
    from google.colab import files
    print("Running in Colab. Upload your three CSV files:")
    print("  - bumble_reddit.csv")
    print("  - bumble_google_play.csv")
    print("  - bumble_app_store.csv")
    uploaded = files.upload()
    print(f"\nUploaded: {list(uploaded.keys())}")
except ImportError:
    print("Not in Colab — reading files from local directory.")

Running in Colab. Upload your three CSV files:
  - bumble_reddit.csv
  - bumble_google_play.csv
  - bumble_app_store.csv


Saving bumble_app_store.csv to bumble_app_store (3).csv
Saving bumble_google_play.csv to bumble_google_play (3).csv
Saving bumble_reddit.csv to bumble_reddit (3).csv

Uploaded: ['bumble_app_store (3).csv', 'bumble_google_play (3).csv', 'bumble_reddit (3).csv']


In [ ]:
# ---------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------

INPUT_REDDIT    = "bumble_reddit.csv"
INPUT_GPLAY     = "bumble_google_play.csv"
INPUT_APPSTORE  = "bumble_app_store.csv"
OUTPUT_COMBINED = "bumble_combined.csv"

COMBINED_COLUMNS = [
    "source",
    "original_date",
    "comment_date",
    "text",
    "comment",
    "subreddit",
    "post_title",
    "post_score",
    "comment_score",
    "rating",
    "thumbs_up",
    "vader_compound",
    "vader_positive",
    "vader_negative",
    "vader_neutral",
    "sentiment_label"
]

print("Configuration loaded.")

Configuration loaded.


In [ ]:
# Validate all three files are present and have correct columns
sources = {
    "Reddit":      INPUT_REDDIT,
    "Google Play": INPUT_GPLAY,
    "App Store":   INPUT_APPSTORE
}

all_ok = True
for name, path in sources.items():
    if os.path.exists(path):
        df_check = pd.read_csv(path)
        missing  = [c for c in COMBINED_COLUMNS if c not in df_check.columns]
        status   = f"{len(df_check)} rows — OK" if not missing else f"{len(df_check)} rows — MISSING COLUMNS: {missing}"
        print(f"{name}: {status}")
        if missing:
            all_ok = False
    else:
        print(f"{name}: FILE NOT FOUND — {path}")
        all_ok = False

if all_ok:
    print("\nAll files present and valid. Ready to combine.")
else:
    print("\nFix missing files before continuing.")

Reddit: 18841 rows — OK
Google Play: 1725 rows — OK
App Store: 489 rows — OK

All files present and valid. Ready to combine.


In [ ]:
df_reddit   = pd.read_csv(INPUT_REDDIT)
df_gplay    = pd.read_csv(INPUT_GPLAY)
df_appstore = pd.read_csv(INPUT_APPSTORE)

# Combine all three
df = pd.concat([df_reddit, df_gplay, df_appstore], ignore_index=True)

# Enforce column order
df = df[COMBINED_COLUMNS]

# Standardise date columns
df["original_date"] = pd.to_datetime(df["original_date"], errors="coerce")
df["comment_date"]  = pd.to_datetime(df["comment_date"],  errors="coerce")

# Sort by original_date descending
df = df.sort_values("original_date", ascending=False).reset_index(drop=True)

print(f"Combined dataset: {len(df)} rows")
print(f"\nBreakdown by source:")
print(df["source"].value_counts().to_string())
print(f"\noriginal_date range: {df['original_date'].min().date()} to {df['original_date'].max().date()}")
print(f"comment_date range:  {df['comment_date'].min().date()} to {df['comment_date'].max().date()}")
df.head()

Combined dataset: 21055 rows

Breakdown by source:
source
reddit         18841
google_play     1725
app_store        489

original_date range: 2023-10-28 to 2026-05-18
comment_date range:  2023-10-28 to 2026-05-18


,source,original_date,comment_date,text,comment,subreddit,post_title,post_score,comment_score,rating,thumbs_up,vader_compound,vader_positive,vader_negative,vader_neutral,sentiment_label
0,reddit,2026-05-18,2026-05-18,If it's not AI: The person who took this photo...,If it's not AI: The person who took this photo...,bumble,(38f) nyc profile review.,1.0,1.0,NaN,NaN,0.4767,0.186,0.000,0.814,positive
1,reddit,2026-05-18,2026-05-18,"**All ""Dating Question"" and ""Hinge Experience""...","**All ""Dating Question"" and ""Hinge Experience""...",hingeapp,28F - Would you feel catfished if you met up w...,1.0,1.0,NaN,NaN,0.8085,0.100,0.021,0.879,positive
2,reddit,2026-05-18,2026-05-18,I’m with you on that one. If you go on a date ...,I’m with you on that one. If you go on a date ...,bumble,Removed after date; People are so weird,1.0,1.0,NaN,NaN,0.3716,0.109,0.000,0.891,positive
3,reddit,2026-05-18,2026-05-18,Okay? \n\nIt doesn't sound like you thought it...,Okay? \n\nIt doesn't sound like you thought it...,bumble,Removed after date; People are so weird,1.0,1.0,NaN,NaN,0.7641,0.256,0.145,0.599,positive
4,reddit,2026-05-18,2026-05-18,Yes there were multiple red flags. Yes there w...,Yes there were multiple red flags. Yes there w...,bumble,Removed after date; People are so weird,1.0,1.0,NaN,NaN,0.8985,0.282,0.152,0.567,positive


In [ ]:
# Data quality report
print("DATA QUALITY REPORT")
print("=" * 50)
print(f"Total rows:            {len(df)}")
print(f"Duplicate text rows:   {df.duplicated(subset=['text', 'source']).sum()}")
print(f"Missing text:          {df['text'].isna().sum()}")
print(f"Missing original_date: {df['original_date'].isna().sum()}")
print(f"Missing comment_date:  {df['comment_date'].isna().sum()} (expected for all app store rows)")
print(f"Missing sentiment:     {df['sentiment_label'].isna().sum()}")
print(f"\nNull counts by field:")
print(df.isna().sum().to_string())

DATA QUALITY REPORT
Total rows:            21055
Duplicate text rows:   2002
Missing text:          0
Missing original_date: 0
Missing comment_date:  2214 (expected for all app store rows)
Missing sentiment:     0

Null counts by field:
source                 0
original_date          0
comment_date        2214
text                   0
comment                0
subreddit           2214
post_title          2214
post_score          2214
comment_score       2214
rating             18841
thumbs_up          19330
vader_compound         0
vader_positive         0
vader_negative         0
vader_neutral          0
sentiment_label        0


In [ ]:
print("SENTIMENT SUMMARY BY SOURCE")
print("=" * 60)
summary = df.groupby("source").agg(
    total_rows   = ("text", "count"),
    avg_compound = ("vader_compound", "mean"),
    avg_rating   = ("rating", "mean"),
    pct_positive = ("sentiment_label", lambda x: (x=="positive").mean()*100),
    pct_neutral  = ("sentiment_label", lambda x: (x=="neutral").mean()*100),
    pct_negative = ("sentiment_label", lambda x: (x=="negative").mean()*100),
).round(2)
print(summary.to_string())
print(f"\nOverall avg compound: {df['vader_compound'].mean():.3f}")
print(f"\nOverall sentiment split:")
print(df["sentiment_label"].value_counts(normalize=True).mul(100).round(1).to_string())
print(f"\nQuarterly sentiment trend (original_date):")
df["quarter"] = df["original_date"].dt.to_period("Q")
print(df.groupby("quarter")["vader_compound"].mean().round(3).to_string())
df = df.drop(columns=["quarter"])

SENTIMENT SUMMARY BY SOURCE
             total_rows  avg_compound  avg_rating  pct_positive  pct_neutral  pct_negative
source                                                                                    
app_store           489         -0.05        1.50         39.26        11.04         49.69
google_play        1725         -0.09        1.66         34.26        15.59         50.14
reddit            18841          0.36         NaN         67.70         9.62         22.68

Overall avg compound: 0.315

Overall sentiment split:
sentiment_label
positive    64.3
negative    25.6
neutral     10.1

Quarterly sentiment trend (original_date):
quarter
2023Q4    0.375
2024Q1    0.401
2024Q2    0.400
2024Q3    0.350
2024Q4    0.376
2025Q1    0.342
2025Q2    0.317
2025Q3    0.341
2025Q4    0.348
2026Q1    0.278
2026Q2    0.095
Freq: Q-DEC


In [ ]:
df.to_csv(OUTPUT_COMBINED, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_COMBINED}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

Saved 21055 rows to bumble_combined.csv
Columns (16): ['source', 'original_date', 'comment_date', 'text', 'comment', 'subreddit', 'post_title', 'post_score', 'comment_score', 'rating', 'thumbs_up', 'vader_compound', 'vader_positive', 'vader_negative', 'vader_neutral', 'sentiment_label']


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_COMBINED)
    print(f"Download triggered for {OUTPUT_COMBINED}")
except ImportError:
    print(f"Not in Colab — file saved locally as {OUTPUT_COMBINED}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered for bumble_combined.csv
